### Cell 1 - Imports and Seeds

In [1]:
import random
import numpy as np
import torch

from dataset_utils.constants import Cresci17SetTypes
from dataset_utils.Cresci17 import Cresci17
from dataset_utils.InterleavedIterableDataset import (
    InterleavedIterableDataset,
)

from feature_pipeline import FeaturePipeline
from experiment_runner import (
    TaskDefinition,
    run_continual_experiment,
)
from classifier_strategies import MulticlassPNNStrategy


RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

c:\Users\Hamouda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


### Cell 2 — Cresci multiclass tasks

In [2]:
DATASET_ROOT = "./datasets"


def identity_label(label: str) -> str:
    return str(label)


# ---------------------------------------------------------
# Task 0:
# genuine_user vs fake_followers
# ---------------------------------------------------------
def make_initial_cresci_dataset(mode: str):
    return InterleavedIterableDataset(
        datasets=[
            Cresci17(
                subset_type=Cresci17SetTypes.GENUINE_USER,
                mode=mode,
                root=DATASET_ROOT,
            ),
            Cresci17(
                subset_type=Cresci17SetTypes.FAKE_FOLLOWER,
                mode=mode,
                root=DATASET_ROOT,
            ),
        ],
        mode="RoundRobin",
    )


# ---------------------------------------------------------
# Grouped Cresci subtypes
# ---------------------------------------------------------
SOCIAL_SPAM_SUBSETS = [
    Cresci17SetTypes.SOCIAL_SPAM_1,
    Cresci17SetTypes.SOCIAL_SPAM_2,
    Cresci17SetTypes.SOCIAL_SPAM_3,
]

TRADITIONAL_SPAM_SUBSETS = [
    Cresci17SetTypes.TRADITIONAL_SPAM_1,
    Cresci17SetTypes.TRADITIONAL_SPAM_2,
    Cresci17SetTypes.TRADITIONAL_SPAM_3,
    Cresci17SetTypes.TRADITIONAL_SPAM_4,
]


def make_grouped_cresci_dataset(
    mode: str,
    subset_types: list,
    grouped_label: str,
):
    """
    Loads several Cresci subtype files but gives all of them
    one shared classifier label.
    """

    return InterleavedIterableDataset(
        datasets=[
            Cresci17(
                subset_type=subset_type,
                mode=mode,
                root=DATASET_ROOT,
                custom_label=grouped_label,
            )
            for subset_type in subset_types
        ],
        mode="RoundRobin",
    )


def make_grouped_cresci_task(
    task_name: str,
    subset_types: list,
    grouped_label: str,
) -> TaskDefinition:
    return TaskDefinition(
        name=task_name,

        train_factory=lambda: make_grouped_cresci_dataset(
            mode="train",
            subset_types=subset_types,
            grouped_label=grouped_label,
        ),

        test_factory=lambda: make_grouped_cresci_dataset(
            mode="test",
            subset_types=subset_types,
            grouped_label=grouped_label,
        ),

        label_transform=identity_label,
    )


# ---------------------------------------------------------
# Final continual-learning sequence
# ---------------------------------------------------------
TASK_DEFINITIONS = [
    TaskDefinition(
        name="Cresci17_Initial_Genuine_vs_FakeFollower",

        train_factory=lambda: make_initial_cresci_dataset(
            "train"
        ),

        test_factory=lambda: make_initial_cresci_dataset(
            "test"
        ),

        label_transform=identity_label,
    ),

    make_grouped_cresci_task(
        task_name="Cresci17_SocialSpam",
        subset_types=SOCIAL_SPAM_SUBSETS,
        grouped_label="social_spam",
    ),

    make_grouped_cresci_task(
        task_name="Cresci17_TraditionalSpam",
        subset_types=TRADITIONAL_SPAM_SUBSETS,
        grouped_label="traditional_spam",
    ),
]


for task_index, task in enumerate(TASK_DEFINITIONS):
    print(f"Task {task_index}: {task.name}")

Task 0: Cresci17_Initial_Genuine_vs_FakeFollower
Task 1: Cresci17_SocialSpam
Task 2: Cresci17_TraditionalSpam


### Configure and run experiment

In [3]:
FEATURE_PIPELINE_CONFIG = {
    "embedding_model": "distilbert-base-uncased",

    "max_tweets_per_user": 20,
    "tweet_batch_size": 32,
    "max_token_length": 128,

    "umap_components": 15,
    "umap_neighbors": 20,
    "umap_min_dist": 0.1,
    "umap_metric": "cosine",

    "random_seed": RANDOM_SEED,
}


STRATEGY_CONFIG = {
    "epochs": 15,
    "learning_rate": 1e-2,
    "dropout_p": 0.1,

    "replay_per_class": 1000,
    "balanced_samples_per_class": 1000,

    "hdbscan_min_cluster_size": 10,
    "hdbscan_current_fraction": 0.80,

    # True = expand when labels show that a new class arrived.
    # HDBSCAN remains logged as a diagnostic.
    "use_intervention_override": True,

    "eval_batch_size": 512,
    "random_seed": RANDOM_SEED,
}


feature_pipeline = FeaturePipeline(
    config=FEATURE_PIPELINE_CONFIG,
    device=device,
)

strategy = MulticlassPNNStrategy()

results = run_continual_experiment(
    task_definitions=TASK_DEFINITIONS,
    feature_pipeline=feature_pipeline,
    strategy=strategy,
    strategy_config=STRATEGY_CONFIG,
    device=device,
)

strategy_state = results["strategy_state"]
experiment_manager = results["experiment_manager"]
all_step_metrics = results["all_step_metrics"]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6213.60it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 0: Cresci17_Initial_Genuine_vs_FakeFollower
Embedding 'Cresci17_Initial_Genuine_vs_FakeFollower/train'...
  0 users
  100 users
  200 users
  300 users
  400 users
  500 users
  600 users
  700 users
  800 users
  900 users
  1000 users
  1100 users
  1200 users
  1300 users
  1400 users
  1500 users
  1600 users
  1700 users
  1800 users
  1900 users
  2000 users
  2100 users
  2200 users
  2300 users
  2400 users
  2500 users
  2600 users
  2700 users
  2800 users
  2900 users
  3000 users
  3100 users
  3200 users
  3300 users
  3400 users
  3500 users
  3600 users
  3700 users
  3800 users
  3900 users
  4000 users
  4100 users
  4200 users
  4300 users
  4400 users
  4500 users
  4600 users
  4700 users
  4800 users
  4900 users
Finished 'Cresci17_Initial_Genuine_vs_FakeFollower/train': (4911, 776), labels=['fake_followers', 'genuine_user']


c:\Users\Hamouda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Current labels: ['fake_followers', 'genuine_user']
Unseen labels: ['fake_followers', 'genuine_user']
HDBSCAN novelty signal: False
Created classifier with 2 outputs.
Actual classifier outputs: 2
Balanced classifier-training counts:
  fake_followers: 1000
  genuine_user: 1000
Epoch [1/15] - Loss: 0.4552
Epoch [2/15] - Loss: 0.2760
Epoch [3/15] - Loss: 0.2621
Epoch [4/15] - Loss: 0.2389
Epoch [5/15] - Loss: 0.2399
Epoch [6/15] - Loss: 0.2468
Epoch [7/15] - Loss: 0.2323
Epoch [8/15] - Loss: 0.2100
Epoch [9/15] - Loss: 0.2089
Epoch [10/15] - Loss: 0.2043
Epoch [11/15] - Loss: 0.2108
Epoch [12/15] - Loss: 0.2090
Epoch [13/15] - Loss: 0.2022
Epoch [14/15] - Loss: 0.2058
Epoch [15/15] - Loss: 0.1924
Loaded best model weights (Best Loss: 0.1924)
Embedding 'Cresci17_Initial_Genuine_vs_FakeFollower/test'...
  0 users
  100 users
  200 users
  300 users
  400 users
  500 users
  600 users
Finished 'Cresci17_Initial_Genuine_vs_FakeFollower/test': (604, 776), labels=['fake_followers', 'genuine_user